# Daily Practice — 2026-11-21 — ML Testing & MLOps: A Metamorphic & Reproducibility Test Suite for a Diagnostic Classifier

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset) —
569 real patient records from digitized fine-needle-aspirate images of breast masses, 30 real-valued
features (radius, texture, perimeter, area, smoothness, concavity, etc., each summarized as `mean`,
`se`, and `worst` across the cell nuclei in an image), binary target (malignant / benign). Loaded via
`sklearn.datasets.load_breast_cancer()` — a genuine clinical dataset, not synthetic.

**Problem statement.** In traditional SQA, a test has a clear oracle: given this input, the correct
output is known in advance, and you assert equality. ML systems break that assumption — for most
inputs you don't *know* the correct label, so "does the model work" can't be answered the same way.
This is the **test oracle problem** for ML, and it's the main reason QA practices don't transfer
directly into ML/AI quality engineering. The practical workaround is a toolkit of *indirect* oracles:

- **Data contract tests** — assert the data pipeline is feeding the model what it expects (schema,
  ranges, nulls), independent of any model.
- **Reproducibility tests** — assert that training is deterministic given a fixed seed, and that
  changing the seed changes results by a *bounded, expected* amount, not wildly.
- **Metamorphic tests** — when you don't know the correct output for input *x*, you can often still
  know a *relation* that must hold between the outputs for *x* and a transformed *x'*. Two concrete
  relations used below:
  - **Invariance**: perturbing a diagnostic measurement by less than realistic instrument noise
    should not flip the diagnosis for more than a small fraction of patients.
  - **Directional expectation**: making a clinically-aggravating feature (e.g. tumor radius) strictly
    larger while holding everything else fixed should never *decrease* the model's predicted
    probability of malignancy.

**What you should produce:**
1. `validate_schema(df)` — a data contract check returning a list of concrete violations (not just
   True/False) for dtypes, nulls, label values, and physically-impossible measurements.
2. `test_reproducibility(...)` — proof that training with a fixed seed is deterministic across
   repeats, and that varying the seed produces test-accuracy values within a small, bounded spread.
3. `test_invariance_to_measurement_noise(...)` — a metamorphic invariance check with a numeric
   pass/fail gate (e.g. "fewer than 2% of predictions may flip").
4. `test_monotonic_direction(...)` — a metamorphic directional check on a probe set of real patient
   rows, reporting the violation rate for a chosen feature.
5. A small **test report** that runs all checks like a test suite and prints PASS/FAIL against
   explicit thresholds — not just numbers with no verdict attached.


---

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
pd.set_option("display.max_columns", 10)


## Load the data

No cleaning is needed — this is a curated toy dataset — but we still run a schema check against it before touching the model, because in a real pipeline the data you get today is never guaranteed to look like the data you validated yesterday.

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df = df.rename(columns={"target": "label"})  # 0 = malignant, 1 = benign (sklearn's own encoding)

FEATURES = list(data.feature_names)
TARGET = "label"

print(df.shape)
df.head()


## Part 1 — Data contract test

Fill in `validate_schema`. It should return a **list of human-readable violation strings** (empty
list = pass). Check at least:
- every column in `FEATURES` is numeric (`float` or `int`) and present
- no nulls anywhere in `FEATURES + [TARGET]`
- `TARGET` only contains values in `{0, 1}`
- every `mean radius`, `mean area`, `mean perimeter` value is strictly positive (a measurement of
  zero or negative size is physically impossible, and would signal a corrupt upstream record)
- the row count is at least 100 (an unexpectedly tiny batch from an upstream feed is itself a defect
  signal, even if every row individually looks fine)


In [ ]:
def validate_schema(frame: pd.DataFrame) -> list[str]:
    # Return a list of violation strings; empty list means the schema check passed.
    violations = []
    # TODO: numeric dtype check for FEATURES
    # TODO: null check for FEATURES + [TARGET]
    # TODO: TARGET in {0, 1} check
    # TODO: positivity check for mean radius / mean area / mean perimeter
    # TODO: minimum row-count check
    return violations


violations = validate_schema(df)
print("Schema violations:", violations)


## Part 2 — Reproducibility test

Fill in `test_reproducibility`. It should:
1. Train the given `model_factory()` (a zero-arg callable returning a fresh unfitted estimator)
   `n_repeats` times on the **same** train/test split with the **same** `random_state`, and confirm
   the resulting test accuracy is identical (within floating-point tolerance, e.g. `1e-12`) every
   time — proof the pipeline has no hidden nondeterminism (unset seeds, dict-ordering bugs, etc.).
2. Train the model with `n_repeats` **different** seeds (vary the train/test split's `random_state`)
   and confirm the spread (max − min) of test accuracy stays under `max_spread` — proof the model
   isn't so unstable that "it works" is a coin flip depending on how the data happened to split.

Return a dict with at least `same_seed_accuracies`, `diff_seed_accuracies`, `same_seed_deterministic`
(bool), `diff_seed_spread` (float), and `diff_seed_within_bound` (bool).


In [ ]:
def test_reproducibility(model_factory, X, y, n_repeats=5, max_spread=0.05):
    # Check same-seed determinism and bounded spread across different seeds.
    same_seed_accuracies = []
    # TODO: repeat n_repeats times with a FIXED random_state for the split, fit model_factory(),
    #       record test accuracy each time

    diff_seed_accuracies = []
    # TODO: repeat n_repeats times with a DIFFERENT random_state per repeat (e.g. 0..n_repeats-1),
    #       record test accuracy each time

    result = {
        "same_seed_accuracies": same_seed_accuracies,
        "diff_seed_accuracies": diff_seed_accuracies,
        "same_seed_deterministic": None,  # TODO: all same_seed_accuracies equal within 1e-12?
        "diff_seed_spread": None,         # TODO: max - min of diff_seed_accuracies
        "diff_seed_within_bound": None,   # TODO: diff_seed_spread <= max_spread?
    }
    return result


def make_model():
    return RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)


repro_result = test_reproducibility(make_model, df[FEATURES], df[TARGET])
repro_result


## Part 3 — Metamorphic invariance test

Fill in `test_invariance_to_measurement_noise`. Train a model once on a train/test split. Then, on
the **test set**, for `n_trials` trials: add independent Gaussian noise to every feature column,
scaled to `noise_std_frac` of that column's own standard deviation (this stands in for realistic
digitizer/measurement imprecision — a `mean radius` reading is not going to be off by 50%, but it
might be off by a fraction of a percent). Compare the noisy predictions to the original predictions
and compute the **flip rate** (fraction of predictions that changed). Average the flip rate across
trials and gate it against `max_flip_rate`.

Return `(mean_flip_rate, passed: bool)`.


In [ ]:
def test_invariance_to_measurement_noise(
    model, X_test, noise_std_frac=0.005, n_trials=20, max_flip_rate=0.02, random_state=RANDOM_STATE
):
    # Metamorphic invariance: tiny measurement noise should rarely flip a prediction.
    rng = np.random.default_rng(random_state)
    original_preds = model.predict(X_test)

    flip_rates = []
    # TODO: for n_trials trials:
    #   - build a noisy copy of X_test: X_test + rng.normal(0, noise_std_frac * X_test.std(), X_test.shape)
    #   - predict on the noisy copy
    #   - compute the fraction of predictions that differ from original_preds
    #   - append that fraction to flip_rates

    mean_flip_rate = None  # TODO: mean of flip_rates
    passed = None          # TODO: mean_flip_rate <= max_flip_rate
    return mean_flip_rate, passed


## Part 4 — Metamorphic directional test

Fill in `test_monotonic_direction`. Pick a probe set: the first `n_probe` rows of the test set.
For each probe row and each step in `step_fracs` (e.g. `[0.05, 0.10, 0.20]`), build a modified copy
of that row where `feature` is increased by `step * feature's std` (holding every other feature
fixed), and compare the model's *predicted probability of malignancy* before and after. A
**violation** is when the probability goes *down* after increasing a feature that is clinically
associated with more aggressive tumors (larger radius/area/concavity → more malignant, never less).

Return the **violation rate** (violations / total comparisons) — in a real test suite you would gate
this at some small nonzero threshold, since this is a plausibility heuristic, not a hard guarantee.


In [ ]:
def test_monotonic_direction(model, X_test, feature, n_probe=30, step_fracs=(0.05, 0.10, 0.20)):
    # Metamorphic directional check: increasing `feature` should not lower P(malignant).
    # sklearn's target encoding here has 0 = malignant, so P(malignant) = predict_proba(...)[:, 0]
    probe = X_test.iloc[:n_probe].copy()
    feature_std = X_test[feature].std()

    violations = 0
    total = 0
    # TODO: for each row in probe, for each step in step_fracs:
    #   - base_proba = model.predict_proba(row as a 1-row frame)[0, 0]
    #   - bumped = row.copy(); bumped[feature] += step * feature_std
    #   - bumped_proba = model.predict_proba(bumped as a 1-row frame)[0, 0]
    #   - total += 1; if bumped_proba < base_proba: violations += 1

    violation_rate = None  # TODO: violations / total
    return violation_rate


## Part 5 — Assemble the test suite

Train a model, run every check above against explicit thresholds, and print a PASS/FAIL report —
the kind of output you'd want in a CI job, not just raw numbers.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES], df[TARGET], test_size=0.25, random_state=RANDOM_STATE, stratify=df[TARGET]
)
model = make_model().fit(X_train, y_train)
print("Holdout accuracy:", accuracy_score(y_test, model.predict(X_test)))

# TODO: call validate_schema(df), test_reproducibility(...), test_invariance_to_measurement_noise(...),
# and test_monotonic_direction(...) (try feature="mean radius" and separately "worst concave points"),
# then print a report like:
#
#   [PASS] schema contract         : 0 violations
#   [PASS] reproducibility (seed)  : deterministic=True, diff-seed spread=0.0123 (<= 0.05)
#   [PASS] invariance (noise)      : mean flip rate=0.0050 (<= 0.02)
#   [FAIL] directional (mean radius): violation rate=0.0667 (investigate before shipping)


### Write-up

TODO: in 3-5 sentences — which of these four checks would catch a regression that a plain
"accuracy on a fixed holdout set" CI gate would miss, and why? Also: the directional test on
`mean radius` or `worst concave points` will very likely show a *nonzero* violation rate even for a
perfectly good linear model — explain why a 0% violation rate isn't the right bar for that test, and
what a reasonable gate threshold should be based on instead.


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
# --- Part 1: schema contract --------------------------------------------
def validate_schema(frame: pd.DataFrame) -> list[str]:
    violations = []

    missing_cols = [c for c in FEATURES if c not in frame.columns]
    if missing_cols:
        violations.append(f"missing feature columns: {missing_cols}")

    present = [c for c in FEATURES if c in frame.columns]
    non_numeric = [c for c in present if not pd.api.types.is_numeric_dtype(frame[c])]
    if non_numeric:
        violations.append(f"non-numeric feature columns: {non_numeric}")

    null_cols = frame[present + [TARGET]].columns[frame[present + [TARGET]].isna().any()].tolist()
    if null_cols:
        violations.append(f"columns containing nulls: {null_cols}")

    bad_labels = set(frame[TARGET].unique()) - {0, 1}
    if bad_labels:
        violations.append(f"label values outside {{0, 1}}: {bad_labels}")

    for col in ["mean radius", "mean area", "mean perimeter"]:
        if col in frame.columns and (frame[col] <= 0).any():
            n_bad = int((frame[col] <= 0).sum())
            violations.append(f"{col}: {n_bad} rows with non-positive values")

    if len(frame) < 100:
        violations.append(f"row count {len(frame)} is below the minimum expected batch size of 100")

    return violations


violations = validate_schema(df)
print("Schema violations:", violations)  # [] on the clean sklearn dataset


# --- Part 2: reproducibility ---------------------------------------------
def test_reproducibility(model_factory, X, y, n_repeats=5, max_spread=0.05):
    same_seed_accuracies = []
    for _ in range(n_repeats):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)
        m = model_factory().fit(X_tr, y_tr)
        same_seed_accuracies.append(accuracy_score(y_te, m.predict(X_te)))

    diff_seed_accuracies = []
    for seed in range(n_repeats):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=seed)
        m = model_factory().fit(X_tr, y_tr)
        diff_seed_accuracies.append(accuracy_score(y_te, m.predict(X_te)))

    diff_seed_spread = max(diff_seed_accuracies) - min(diff_seed_accuracies)
    return {
        "same_seed_accuracies": same_seed_accuracies,
        "diff_seed_accuracies": diff_seed_accuracies,
        "same_seed_deterministic": max(same_seed_accuracies) - min(same_seed_accuracies) < 1e-12,
        "diff_seed_spread": diff_seed_spread,
        "diff_seed_within_bound": diff_seed_spread <= max_spread,
    }


def make_model():
    return RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)


repro_result = test_reproducibility(make_model, df[FEATURES], df[TARGET])
repro_result


# --- Part 3: metamorphic invariance --------------------------------------
def test_invariance_to_measurement_noise(
    model, X_test, noise_std_frac=0.005, n_trials=20, max_flip_rate=0.02, random_state=RANDOM_STATE
):
    rng = np.random.default_rng(random_state)
    original_preds = model.predict(X_test)

    flip_rates = []
    col_std = X_test.std()
    for _ in range(n_trials):
        noise = rng.normal(0.0, noise_std_frac * col_std.values, X_test.shape)
        noisy_X = X_test + noise
        noisy_preds = model.predict(noisy_X)
        flip_rates.append(float(np.mean(noisy_preds != original_preds)))

    mean_flip_rate = float(np.mean(flip_rates))
    passed = mean_flip_rate <= max_flip_rate
    return mean_flip_rate, passed


# --- Part 4: metamorphic directional check -------------------------------
def test_monotonic_direction(model, X_test, feature, n_probe=30, step_fracs=(0.05, 0.10, 0.20)):
    probe = X_test.iloc[:n_probe].copy()
    feature_std = X_test[feature].std()

    violations = 0
    total = 0
    for _, row in probe.iterrows():
        base_proba = model.predict_proba(row.to_frame().T)[0, 0]
        for step in step_fracs:
            bumped = row.copy()
            bumped[feature] = bumped[feature] + step * feature_std
            bumped_proba = model.predict_proba(bumped.to_frame().T)[0, 0]
            total += 1
            if bumped_proba < base_proba:
                violations += 1

    return violations / total


# --- Part 5: assemble and report -----------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES], df[TARGET], test_size=0.25, random_state=RANDOM_STATE, stratify=df[TARGET]
)
model = make_model().fit(X_train, y_train)
print("Holdout accuracy:", accuracy_score(y_test, model.predict(X_test)))

schema_violations = validate_schema(df)
repro = test_reproducibility(make_model, df[FEATURES], df[TARGET])
flip_rate, invariance_passed = test_invariance_to_measurement_noise(model, X_test)
radius_violation_rate = test_monotonic_direction(model, X_test, "mean radius")
concave_violation_rate = test_monotonic_direction(model, X_test, "worst concave points")

def verdict(ok):
    return "PASS" if ok else "FAIL"

print(f"[{verdict(len(schema_violations) == 0)}] schema contract         : "
      f"{len(schema_violations)} violations")
print(f"[{verdict(repro['same_seed_deterministic'] and repro['diff_seed_within_bound'])}] "
      f"reproducibility (seed)  : deterministic={repro['same_seed_deterministic']}, "
      f"diff-seed spread={repro['diff_seed_spread']:.4f} (<= 0.05)")
print(f"[{verdict(invariance_passed)}] invariance (noise)      : "
      f"mean flip rate={flip_rate:.4f} (<= 0.02)")
print(f"[{verdict(radius_violation_rate <= 0.10)}] directional (mean radius): "
      f"violation rate={radius_violation_rate:.4f}")
print(f"[{verdict(concave_violation_rate <= 0.10)}] directional (worst concave pts): "
      f"violation rate={concave_violation_rate:.4f}")
```

**Why this matters:** a fixed-holdout accuracy gate only tells you the model was good *on that one
frozen sample* at *that one point in training*. It says nothing about whether training is
deterministic (you could ship a "90% accuracy" model that was actually a lucky seed), whether the
model is brittle to realistic measurement noise it will see in production, or whether it has learned
a sane relationship between a clinically meaningful feature and its output — all of which the
metamorphic and reproducibility checks catch directly, with no labeled oracle required.

**Why the directional violation rate shouldn't be gated at 0%:** a random forest's decision function
is a step function built from per-tree thresholds, not a smooth monotonic curve — nothing in how it's
trained *guarantees* that `P(malignant)` moves one way as `mean radius` increases, it's just usually
true because that's the real-world relationship in the training data. `mean radius` is also
correlated with, but not deterministically fixed by, the other 29 features, so bumping it in
isolation can push a probe point into a region of feature space the forest never saw associated with
that specific combination, and a handful of trees can legitimately flip their vote near a threshold.
A reasonable gate is set empirically — e.g. run the check against several known-good historical model
versions, take the worst observed violation rate across them, and gate new models against that
baseline plus a small margin, rather than against a theoretical 0%.

</details>